# WhisperSubs Colab
Japanese audio → casual Indonesian SRT using kotoba-whisper-v2.0 (float16) + GPT-4o

> **Before running:** Runtime → Change runtime type → T4 GPU
>
> **Workflow:** Run all cells top to bottom. Enter API key when prompted. Upload audio. Download SRT.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch torchaudio silero-vad soundfile openai
print('Dependencies installed.')

In [ ]:
# Cell 2: Configuration — run this once per session
import getpass
import os

OPENAI_API_KEY = getpass.getpass("OpenAI API Key: ")
GPT_MODEL = "gpt-4o"
BATCH_SIZE = 5
USE_VAD = True   # Set False to skip VAD and feed full audio directly

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
vad_label = f"True (silero-vad)" if USE_VAD else "False (disabled)"
print(f"Config set.")
print(f"  GPT model  : {GPT_MODEL}")
print(f"  Whisper    : kotoba-tech/kotoba-whisper-v2.0")
print(f"  VAD filter : {vad_label}")
print(f"  Batch size : {BATCH_SIZE} segments per GPT call")

In [ ]:
# Cell 3: Upload audio file
from google.colab import files

print("Select your audio file (MP3, WAV, M4A, FLAC, etc.):")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Re-run this cell and select a file.")

audio_filename = list(uploaded.keys())[0]
file_size_mb = len(uploaded[audio_filename]) / 1024 / 1024
print(f"Ready: {audio_filename} ({file_size_mb:.1f} MB)")

In [ ]:
# Cell 4: Transcribe audio
# First run: downloads model to Colab (~3GB, 3-5 min). Cached for this session.
import torch
import torchaudio
import soundfile as sf
import numpy as np
from transformers import pipeline

# VAD parameters — tune here without touching Cell 2
VAD_THRESHOLD      = 0.2
VAD_MIN_SPEECH_MS  = 100
VAD_MIN_SILENCE_MS = 200
VAD_MERGE_GAP_MS   = 200
SAMPLE_RATE        = 16000
MODEL_ID           = "kotoba-tech/kotoba-whisper-v2.0"

print(f"Loading model: {MODEL_ID} on T4 GPU (float16)...")
print("(First run downloads ~3GB — takes 3-5 min. Cached for this session.)")

if 'pipe' not in dir():
    pipe = pipeline(
        "automatic-speech-recognition",
        model=MODEL_ID,
        torch_dtype=torch.float16,
        device="cuda",
    )

# Load audio as float32 mono at 16kHz
print("Loading audio...")
audio_data, sr = sf.read(audio_filename, dtype="float32")
if audio_data.ndim > 1:
    audio_data = audio_data.mean(axis=1)
if sr != SAMPLE_RATE:
    audio_tensor = torch.tensor(audio_data).unsqueeze(0)
    audio_data = torchaudio.functional.resample(audio_tensor, sr, SAMPLE_RATE).squeeze(0).numpy()

duration = len(audio_data) / SAMPLE_RATE
print(f"Audio loaded: {duration:.1f}s")

# Determine speech spans
if USE_VAD:
    print("Running silero-vad...")
    vad_model, utils = torch.hub.load(
        repo_or_dir="snakers4/silero-vad",
        model="silero_vad",
        force_reload=False,
        verbose=False,
        trust_repo=True,
    )
    get_speech_timestamps = utils.get_speech_timestamps

    audio_tensor = torch.tensor(audio_data)
    raw_spans = get_speech_timestamps(
        audio_tensor,
        vad_model,
        threshold=VAD_THRESHOLD,
        min_speech_duration_ms=VAD_MIN_SPEECH_MS,
        min_silence_duration_ms=VAD_MIN_SILENCE_MS,
        sampling_rate=SAMPLE_RATE,
        return_seconds=True,
    )

    if not raw_spans:
        print("WARNING: VAD found no speech. Try lowering VAD_THRESHOLD or setting USE_VAD = False.")

    # Merge spans closer than VAD_MERGE_GAP_MS
    spans = []
    for span in raw_spans:
        if spans and (span["start"] - spans[-1]["end"]) < VAD_MERGE_GAP_MS / 1000:
            spans[-1]["end"] = span["end"]
        else:
            spans.append({"start": span["start"], "end": span["end"]})

    print(f"VAD found {len(spans)} speech spans (merged from {len(raw_spans)} raw).")
else:
    spans = [{"start": 0.0, "end": duration}]
    print("VAD disabled — transcribing full audio as one span.")

# Transcribe each span
print("Transcribing...")
segments = []
for i, span in enumerate(spans):
    start_sample = int(span["start"] * SAMPLE_RATE)
    end_sample   = int(span["end"]   * SAMPLE_RATE)
    audio_slice  = audio_data[start_sample:end_sample]

    if len(audio_slice) == 0:
        continue

    result = pipe(
        {"array": audio_slice, "sampling_rate": SAMPLE_RATE},
        chunk_length_s=30,
        batch_size=8,   # internal Whisper chunk batching — unrelated to GPT BATCH_SIZE
        return_timestamps=True,
        generate_kwargs={"language": "japanese", "task": "transcribe"},
    )

    for chunk in result["chunks"]:
        ts = chunk["timestamp"]
        if ts[0] is None or ts[1] is None:
            continue
        text = chunk["text"].strip()
        if not text:
            continue
        end_ts = min(span["start"] + ts[1], span["end"])
        segments.append({
            "start": span["start"] + ts[0],
            "end":   end_ts,
            "text":  text,
        })

    if (i + 1) % 10 == 0 or (i + 1) == len(spans):
        print(f"  Processed {i + 1}/{len(spans)} spans...")

print(f"\nTranscription complete! Total segments: {len(segments)}")
if segments:
    print(f"  First: [{segments[0]['start']:.2f}s → {segments[0]['end']:.2f}s] {segments[0]['text'][:80]}")
    print(f"  Last:  [{segments[-1]['start']:.2f}s → {segments[-1]['end']:.2f}s] {segments[-1]['text'][:80]}")

In [ ]:
# Cell 4b: Export raw Japanese transcription as SRT (timing check — no translation needed)
from google.colab import files

def format_time(seconds):
    total_ms = round(float(seconds) * 1000)
    ms = total_ms % 1000
    total_secs = total_ms // 1000
    hours, remainder = divmod(total_secs, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"

srt_raw = ""
for i, seg in enumerate(segments):
    srt_raw += f"{i + 1}\n{format_time(seg['start'])} --> {format_time(seg['end'])}\n{seg['text']}\n\n"

base_name = audio_filename.rsplit(".", 1)[0]
raw_filename = f"{base_name}_ja.srt"

with open(raw_filename, "w", encoding="utf-8") as f:
    f.write(srt_raw)

print(f"Japanese SRT saved: {raw_filename} ({len(segments)} segments)")
print("\nFirst 5 segments:")
for seg in segments[:5]:
    print(f"  [{format_time(seg['start'])} --> {format_time(seg['end'])}] {seg['text']}")

files.download(raw_filename)
print("Download triggered.")

In [ ]:
# Cell 5: Translate segments to casual Indonesian via GPT-4o
import re
import time
from openai import OpenAI

TRANSLATION_SYSTEM_PROMPT = (
    "Kamu adalah penerjemah subtitle dari bahasa Jepang ke bahasa Indonesia. "
    "Terjemahkan setiap dialog dalam tanda [Dialog X] ke bahasa Indonesia.\n\n"
    "Aturan gaya bahasa:\n"
    "- Gunakan \"aku/kamu\" bukan \"saya/Anda\"\n"
    "- Pakai bahasa sehari-hari yang santai seperti ngobrol sama teman\n"
    "- Hindari bahasa baku/formal (jangan pakai \"telah\", \"namun\", \"dapat\", dll)\n"
    "- Hindari slang berat Jakarta (jangan pakai \"gue/lu\", \"anjir\", dll)\n"
    "- Singkat dan natural, seperti subtitle anime fansub\n"
    "- Jaga konteks antar dialog agar cerita tetap nyambung\n\n"
    "Pertahankan format [Dialog X] agar bisa dicocokkan kembali."
)

client = OpenAI(api_key=OPENAI_API_KEY)
translated_segments = []
total_batches = (len(segments) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_start in range(0, len(segments), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(segments))
    batch_segments = segments[batch_start:batch_end]
    batch_num = batch_start // BATCH_SIZE + 1

    print(f"Batch {batch_num}/{total_batches} — segments {batch_start + 1}-{batch_end}...")

    batch_texts = []
    for i, seg in enumerate(batch_segments):
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', seg['text'])
        batch_texts.append(f"[Dialog {i + 1}] {text}")

    combined_text = "\n".join(batch_texts)

    try:
        response = client.chat.completions.create(
            model=GPT_MODEL,
            messages=[
                {"role": "system", "content": TRANSLATION_SYSTEM_PROMPT},
                {"role": "user", "content": combined_text},
            ],
            temperature=0.6,
        )

        result_text = response.choices[0].message.content.strip()

        # Parse [Dialog N] markers out of GPT response
        translated_dialogs = {}
        current_dialog = None
        current_text = []

        for line in result_text.split('\n'):
            if line.strip().startswith('[Dialog'):
                if current_dialog is not None:
                    translated_dialogs[current_dialog] = ' '.join(current_text).strip()
                try:
                    dialog_num = int(line.split(']')[0].split()[-1])
                    current_dialog = dialog_num
                    text_after = line.split(']', 1)[1].strip() if ']' in line else ''
                    current_text = [text_after] if text_after else []
                except Exception:
                    continue
            elif current_dialog is not None and line.strip():
                current_text.append(line.strip())

        if current_dialog is not None:
            translated_dialogs[current_dialog] = ' '.join(current_text).strip()

        for i, seg in enumerate(batch_segments):
            translated = translated_dialogs.get(i + 1, '')
            if not translated:
                print(f"  Warning: Dialog {batch_start + i + 1} failed — using original JP text")
                translated = seg['text']
            translated_segments.append({
                'start': seg['start'],
                'end': seg['end'],
                'text': translated,
            })

        if batch_end < len(segments):
            time.sleep(1)

    except Exception as e:
        print(f"  Error in batch {batch_num}: {e}")
        for seg in batch_segments:
            translated_segments.append(seg)

print(f"\nTranslation complete! {len(translated_segments)} segments.")
if translated_segments:
    print(f"Sample translation: {translated_segments[0]['text']}")

In [ ]:
# Cell 6: Generate SRT file and download
if 'translated_segments' not in dir() or not translated_segments:
    raise RuntimeError("Run Cell 5 first to generate translations.")

from google.colab import files

def format_time(seconds):
    total_ms = round(float(seconds) * 1000)
    ms = total_ms % 1000
    total_secs = total_ms // 1000
    hours, remainder = divmod(total_secs, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"

srt_content = ""
for i, segment in enumerate(translated_segments):
    start_time = format_time(segment["start"])
    end_time = format_time(segment["end"])
    srt_content += f"{i + 1}\n{start_time} --> {end_time}\n{segment['text'].strip()}\n\n"

base_name = audio_filename.rsplit(".", 1)[0]
output_filename = f"{base_name}_id.srt"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(srt_content)

print(f"SRT saved: {output_filename} ({len(translated_segments)} segments)")
print("Sample (first 3 entries):")
print(srt_content[:400])
files.download(output_filename)
print("Download triggered — check your browser downloads.")